<a href="https://colab.research.google.com/github/Narawit007/BSC_DPDM2025/blob/main/model_comparison_2_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Decision Tree vs Perceptron
**Dataset:** ทำนายฝน (Rainfall) — Season, Humidity, Cloud Cover  
**Train:** Day 1–10 &nbsp;|&nbsp; **Test:** Day 11–13

## Section 1 : โหลด Library และข้อมูล

In [1]:
import pandas as pd
import numpy as np
from math import log2
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.linear_model import Perceptron
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_excel('ข้อสอบ.xlsx')       # เปลี่ยนชื่อไฟล์ตามจริง
df.columns = df.columns.str.strip()
df

,Split,Day,Season,Humidity,Cloud Cover,Rainfall
0,train,1,rainy,high,heavy,rain
1,train,2,rainy,high,heavy,rain
2,train,3,rainy,high,moderate,rain
3,train,4,hot,low,clear,no rain
4,train,5,hot,low,clear,no rain
5,train,6,rainy,high,heavy,rain
6,train,7,rainy,-,heavy,rain
7,train,8,cool,medium,moderate,rain
8,train,9,hot,low,-,no rain
9,train,10,cool,medium,moderate,rain


## Section 2 : เตรียมข้อมูล (Data Preparation)

In [2]:
df_clean = df.copy()
df_clean.replace('-', np.nan, inplace=True)

feature_cols = ['Season', 'Humidity', 'Cloud Cover']

# fillna ด้วย mode ของ train set (กัน data leakage)
for col in feature_cols:
    mode_val = df_clean[df_clean['Split'] == 'train'][col].mode()[0]
    df_clean[col] = df_clean[col].fillna(mode_val)
    print(f"  fillna '{col}' -> '{mode_val}'")

print()
df_clean[['Split','Day'] + feature_cols + ['Rainfall']]

  fillna 'Season' -> 'rainy'
  fillna 'Humidity' -> 'high'
  fillna 'Cloud Cover' -> 'heavy'



,Split,Day,Season,Humidity,Cloud Cover,Rainfall
0,train,1,rainy,high,heavy,rain
1,train,2,rainy,high,heavy,rain
2,train,3,rainy,high,moderate,rain
3,train,4,hot,low,clear,no rain
4,train,5,hot,low,clear,no rain
5,train,6,rainy,high,heavy,rain
6,train,7,rainy,high,heavy,rain
7,train,8,cool,medium,moderate,rain
8,train,9,hot,low,heavy,no rain
9,train,10,cool,medium,moderate,rain


In [3]:
# -----------------------------------------------------------
# Ordinal Encoding (ระบุ order ให้ตรงความหมาย)
# ใช้แทน LabelEncoder เพราะ LabelEncoder เรียงตาม alphabet
# ทำให้ Decision Tree ตัดสินใจผิดพลาดได้
# -----------------------------------------------------------
oe = OrdinalEncoder(categories=[
    ['cool', 'rainy', 'hot'],        # Season   : cool=0 rainy=1 hot=2
    ['low',  'medium', 'high'],      # Humidity : low=0  medium=1 high=2
    ['clear','moderate','heavy'],    # Cloud Cover: clear=0 moderate=1 heavy=2
])
le_y = LabelEncoder()   # Target: no rain=0, rain=1

train_df = df_clean[df_clean['Split'] == 'train']
test_df  = df_clean[df_clean['Split'] == 'test']

oe.fit(train_df[feature_cols])
le_y.fit(train_df['Rainfall'])

X_train = oe.transform(train_df[feature_cols])
y_train = le_y.transform(train_df['Rainfall'])
X_test  = oe.transform(test_df[feature_cols])
y_test  = le_y.transform(test_df['Rainfall'])

print('Ordinal mapping:')
for col, cats in zip(feature_cols, oe.categories_):
    print(f'  {col}: {list(enumerate(cats))}')
print(f'  Rainfall: {dict(enumerate(le_y.classes_))}')

Ordinal mapping:
  Season: [(0, 'cool'), (1, 'rainy'), (2, 'hot')]
  Humidity: [(0, 'low'), (1, 'medium'), (2, 'high')]
  Cloud Cover: [(0, 'clear'), (1, 'moderate'), (2, 'heavy')]
  Rainfall: {0: 'no rain', 1: 'rain'}


## Section 3 : แบ่ง Train / Test

In [4]:
print(f'Train : {X_train.shape[0]} rows  -> Day {list(train_df["Day"].values)}')
print(f'Test  : {X_test.shape[0]}  rows  -> Day {list(test_df["Day"].values)}')
print(f'Features : {feature_cols}')
print(f'Target   : Rainfall  {dict(enumerate(le_y.classes_))}')

Train : 10 rows  -> Day [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]
Test  : 3  rows  -> Day [np.int64(11), np.int64(12), np.int64(13)]
Features : ['Season', 'Humidity', 'Cloud Cover']
Target   : Rainfall  {0: 'no rain', 1: 'rain'}


## Section 3.5 : ตรวจสอบ Information Gain ของแต่ละ Feature
เพื่อยืนยันว่า Decision Tree เลือก root node ถูกต้อง

In [5]:
def entropy(p, n):
    total = p + n
    if total == 0 or p == 0 or n == 0:
        return 0.0
    return -(p/total)*log2(p/total) - (n/total)*log2(n/total)

y = y_train
X = X_train
total  = len(y)
p_yes  = int(y.sum())
p_no   = total - p_yes
info_d = entropy(p_yes, p_no)

print(f'Info(D) = I({p_yes},{p_no}) = {info_d:.6f}')
print()

gains = {}
for i, col in enumerate(feature_cols):
    vals   = sorted(set(X[:, i]))
    info_i = 0
    print(f'--- {col} ---')
    for v in vals:
        mask  = X[:, i] == v
        sub_y = y[mask]
        p = int(sub_y.sum());  n = len(sub_y) - p
        w = len(sub_y) / total
        e = entropy(p, n)
        info_i += w * e
        label = oe.categories_[i][int(v)]
        print(f'  {label:10s}: p={p} n={n}  I={e:.6f}  weight={w:.4f}')
    gain = info_d - info_i
    gains[col] = gain
    print(f'  Gain({col}) = {info_d:.6f} - {info_i:.6f} = {gain:.6f}')
    print()

best = max(gains, key=gains.get)
print(f'=> Root node ที่ถูกต้อง = "{best}" (Gain สูงสุด = {gains[best]:.6f})')

Info(D) = I(7,3) = 0.881291

--- Season ---
  cool      : p=2 n=0  I=0.000000  weight=0.2000
  rainy     : p=5 n=0  I=0.000000  weight=0.5000
  hot       : p=0 n=3  I=0.000000  weight=0.3000
  Gain(Season) = 0.881291 - 0.000000 = 0.881291

--- Humidity ---
  low       : p=0 n=3  I=0.000000  weight=0.3000
  medium    : p=2 n=0  I=0.000000  weight=0.2000
  high      : p=5 n=0  I=0.000000  weight=0.5000
  Gain(Humidity) = 0.881291 - 0.000000 = 0.881291

--- Cloud Cover ---
  clear     : p=0 n=2  I=0.000000  weight=0.2000
  moderate  : p=3 n=0  I=0.000000  weight=0.3000
  heavy     : p=4 n=1  I=0.721928  weight=0.5000
  Gain(Cloud Cover) = 0.881291 - 0.360964 = 0.520327

=> Root node ที่ถูกต้อง = "Season" (Gain สูงสุด = 0.881291)


## Section 4 : โมเดล 1 — Decision Tree
`max_depth=3`, `min_samples_leaf=2`, `criterion='entropy'` (Information Gain)

In [6]:
dt_model = DecisionTreeClassifier(
    max_depth        = 3,
    min_samples_leaf = 2,
    criterion        = 'entropy',
    random_state     = 42
)
dt_model.fit(X_train, y_train)

print('=== โครงสร้างต้นไม้ ===')
print(export_text(dt_model, feature_names=feature_cols))

# แสดง feature importance
print('Feature Importance (Information Gain):')
for col, imp in zip(feature_cols, dt_model.feature_importances_):
    print(f'  {col:15s}: {imp:.4f}')

=== โครงสร้างต้นไม้ ===
|--- Humidity <= 0.50
|   |--- class: 0
|--- Humidity >  0.50
|   |--- class: 1

Feature Importance (Information Gain):
  Season         : 0.0000
  Humidity       : 1.0000
  Cloud Cover    : 0.0000


In [7]:
dt_pred       = dt_model.predict(X_test)
dt_pred_label = le_y.inverse_transform(dt_pred)
y_test_label  = le_y.inverse_transform(y_test)

print('=== ผลการทำนาย Test Set ===')
for i, (pred, actual) in enumerate(zip(dt_pred_label, y_test_label)):
    day    = test_df.iloc[i]['Day']
    status = '✓ ถูก' if pred == actual else '✗ ผิด'
    print(f'  Day {int(day):2d} -> ทำนาย: {pred:8s}  จริง: {actual:8s}  {status}')

print(f'\nAccuracy : {accuracy_score(y_test, dt_pred):.2f}')
print('\nClassification Report:')
print(classification_report(y_test, dt_pred, target_names=le_y.classes_))
print('Confusion Matrix (rows=จริง, cols=ทำนาย):')
print(confusion_matrix(y_test, dt_pred))

=== ผลการทำนาย Test Set ===
  Day 11 -> ทำนาย: rain      จริง: rain      ✓ ถูก
  Day 12 -> ทำนาย: no rain   จริง: no rain   ✓ ถูก
  Day 13 -> ทำนาย: rain      จริง: rain      ✓ ถูก

Accuracy : 1.00

Classification Report:
              precision    recall  f1-score   support

     no rain       1.00      1.00      1.00         1
        rain       1.00      1.00      1.00         2

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3

Confusion Matrix (rows=จริง, cols=ทำนาย):
[[1 0]
 [0 2]]


## Section 5 : โมเดล 2 — Perceptron
`max_iter=2` (2 Epoch), `eta0=0.5` (Learning Rate)

In [8]:
pt_model = Perceptron(
    max_iter     = 2,
    eta0         = 0.5,
    tol          = None,
    random_state = 42
)
pt_model.fit(X_train, y_train)

print('=== Weights หลัง 2 Epoch ===')
print(f'  Bias (w0)                : {pt_model.intercept_[0]:.4f}')
for i, (col, w) in enumerate(zip(feature_cols, pt_model.coef_[0])):
    print(f'  Weight w{i+1} ({col:12s}): {w:.4f}')

=== Weights หลัง 2 Epoch ===
  Bias (w0)                : -0.5000
  Weight w1 (Season      ): -2.0000
  Weight w2 (Humidity    ): 2.0000
  Weight w3 (Cloud Cover ): 0.0000


In [9]:
pt_pred       = pt_model.predict(X_test)
pt_pred_label = le_y.inverse_transform(pt_pred)

print('=== ผลการทำนาย Test Set ===')
for i, (pred, actual) in enumerate(zip(pt_pred_label, y_test_label)):
    day    = test_df.iloc[i]['Day']
    status = '✓ ถูก' if pred == actual else '✗ ผิด'
    print(f'  Day {int(day):2d} -> ทำนาย: {pred:8s}  จริง: {actual:8s}  {status}')

print(f'\nAccuracy : {accuracy_score(y_test, pt_pred):.2f}')
print('\nClassification Report:')
print(classification_report(y_test, pt_pred, target_names=le_y.classes_))
print('Confusion Matrix (rows=จริง, cols=ทำนาย):')
print(confusion_matrix(y_test, pt_pred))

=== ผลการทำนาย Test Set ===
  Day 11 -> ทำนาย: rain      จริง: rain      ✓ ถูก
  Day 12 -> ทำนาย: no rain   จริง: no rain   ✓ ถูก
  Day 13 -> ทำนาย: rain      จริง: rain      ✓ ถูก

Accuracy : 1.00

Classification Report:
              precision    recall  f1-score   support

     no rain       1.00      1.00      1.00         1
        rain       1.00      1.00      1.00         2

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3

Confusion Matrix (rows=จริง, cols=ทำนาย):
[[1 0]
 [0 2]]


## Section 6 : สรุปเปรียบเทียบ 2 โมเดล

In [10]:
summary = pd.DataFrame({
    'โมเดล'    : ['Decision Tree (depth=3, leaf=2)', 'Perceptron (2 Epoch, lr=0.5)'],
    'Encoding' : ['OrdinalEncoder', 'OrdinalEncoder'],
    'Accuracy' : [f'{accuracy_score(y_test, dt_pred):.2f}',
                  f'{accuracy_score(y_test, pt_pred):.2f}']
})
print(summary.to_string(index=False))

                          โมเดล       Encoding Accuracy
Decision Tree (depth=3, leaf=2) OrdinalEncoder     1.00
   Perceptron (2 Epoch, lr=0.5) OrdinalEncoder     1.00
